# Black-Gold — a quantitative teardown 🔬
### The predictive regression with a t-stat · timing vs buy-and-hold · sub-period slopes

![Signal: None](https://img.shields.io/badge/Signal-None-c0392b?style=flat-square)
![Tradability: Mirage](https://img.shields.io/badge/Tradability-Mirage-c0392b?style=flat-square)
![Replicates out of sample?: Busted](https://img.shields.io/badge/Replicates_out_of_sample%3F-Busted-8b949e?style=flat-square)

The deep companion to the [notebook for the curious](01_for_the_curious.ipynb). We test oil→equity predictability on all tradable data and find none.

> ⚠️ **Not investment advice.** WTI (CL=F) + S&P 500 (^GSPC) + 13-week T-bill (^IRX) monthly, 2000-09 → 2026-05, 309 months (Yahoo). **Daily** closes resampled to month-end — Yahoo's native monthly CL=F feed is missing 89 of 310 months, which silently turns a positional one-month lag into a 2–3-month lag; our grid is asserted hole-free and the lag is calendar (`st.lag_one_month`). CL=F begins 2000, so Driesprong's 1973–2003 window isn't testable here — but the value is out-of-sample survival. Sources in [`docs/references.md`](../docs/references.md).

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath("../../.."))  # repo root (quantlab/)
sys.path.insert(0, os.path.abspath(".."))        # study package (black_gold/)
%matplotlib inline
import matplotlib.pyplot as plt
plt.rcParams["figure.figsize"] = (10, 5.5); plt.rcParams["axes.grid"] = True
import numpy as np, pandas as pd
from black_gold import data, strategy as st
from quantlab import repro
d = repro.as_of(data.fetch_pair())             # cache-first (examples/verify.py --fetch); hole-free monthly grid, pinned as-of
reg = st.predict_regression(d["oil"], d["eq"])
timing = st.oil_timing(d["oil"], d["eq"], d["tbill"])  # cash leg earns the 13-week T-bill
bh = st.buy_hold(d["eq"])


## Verdict, up front

| Axis | Stamp | Why |
|---|---|---|
| Signal | **None** | slope −0.001, t −0.03 — exactly zero |
| Tradability | **Mirage** | timing Sharpe 0.32 < buy-and-hold 0.37 (excess of T-bill, both legs) |
| Replicates OOS? | **Busted** | insignificant in 2000–2008 (t −0.9) and 2009-on (t +0.2) |

> 💡 *In plain words:* a documented predictor that vanished out of sample.

## 1 · The claim, steelmanned

- **H₁:** the slope of equity_t on oil_(t−1) is negative and significant.
- **H₂:** the oil-timing rule beats buy-and-hold.
- **H₃:** it holds across sub-periods.

## 2 · So what? — what rides on each

If H₁/H₂ hold, a public price forecasts the market — a tradable cross-asset edge. If they fail OOS, it's a 1973–2003 in-sample artifact.

## 3 · How we'd know — the protocol

OLS equity_t ~ oil_(t−1) with an analytic t-stat and a **calendar** one-month lag on a hole-free monthly grid → timing (T-bill credited in cash) vs buy-and-hold, Sharpe in **excess of the T-bill on both legs** → 2000–2008 / 2009-on slope split.

## 4 · The teardown

### 4.1 The predictive regression (calendar lag, n = 308)

In [2]:
print({k:(round(v,3) if isinstance(v,float) else v) for k,v in reg.items()})
print(f"slope {reg['slope']:+.3f} (Driesprong: negative), t = {reg['tstat']:+.2f} (need |t|>2)")

{'slope': -0.001, 'r': -0.002, 'tstat': -0.031, 'n': 308}
slope -0.001 (Driesprong: negative), t = -0.03 (need |t|>2)


> 💡 *In plain words:* t = −0.03 — a slope of exactly zero. Last month's oil carries no information about this month's stocks. **H₁ rejected.** (An earlier cut of this study, on Yahoo's gapped monthly feed, read +0.017 "wrong sign" — that number was the mis-lag artifact, not the market.)

### 4.2 Timing vs buy-and-hold — T-bill in cash, excess-of-T-bill Sharpe on both legs

In [3]:
display(pd.DataFrame({'oil timing':st.summary(timing, rf=d['tbill']),'buy & hold':st.summary(bh, rf=d['tbill'])}).T[['cagr','sharpe','vol_ann','max_drawdown']].round(3))
print('time in market:', f"{st.time_in_market(d['oil']):.0%}", '| cash leg earns ^IRX | Sharpe = excess of T-bill, like-for-like')

,cagr,sharpe,vol_ann,max_drawdown
oil timing,0.048,0.317,0.113,-0.440
buy & hold,0.064,0.370,0.152,-0.526


time in market: 44% | cash leg earns ^IRX | Sharpe = excess of T-bill, like-for-like


> 💡 *In plain words:* even paid the T-bill while parked, the timer earns less (CAGR 4.8% vs 6.5%) at a lower like-for-like Sharpe (0.32 vs 0.37), out of the market 56% of the time. **H₂ rejected.**

### 4.3 Sub-period slopes

In [4]:
for lab,sl in [('2000-2008',d[d.index.year<=2008]),('2009-on',d[d.index.year>=2009])]:
    rr=st.predict_regression(sl['oil'],sl['eq']); print(f"{lab}: slope {rr['slope']:+.3f}  t={rr['tstat']:+.2f}  n={rr['n']}")

2000-2008: slope -0.042  t=-0.91  n=99
2009-on: slope +0.006  t=+0.22  n=208


> 💡 *In plain words:* 2000–2008 leans the right (negative) way but at t = −0.9 it's noise; 2009-on flips positive and stays dead. **H₃ rejected** — never there in the tradable era.

## 5 · The verdict

H₁, H₂, H₃ all rejected → Signal `NONE`, Tradability `MIRAGE`, out-of-sample replication `BUSTED`.

## 6 · Could you trade it?

No edge to trade. The oil timer underperforms buy-and-hold and adds whipsaw. If oil matters for equities, it's contemporaneous and already priced — not a one-month-ahead signal.

## 7 · Going further

Forks: (a) a longer WTI spot series (pre-2000) to check the original window; (b) oil *shocks* (structural decomposition, Kilian 2009) vs raw returns; (c) sector-level (energy vs the rest) where a contemporaneous oil beta is real. Backlog: [`docs/pwb_strategies_inventory.md`](../../../docs/pwb_strategies_inventory.md).